# MCTS sweep --- results explorer

Two modules do everything: **`dataset`** (build/load the parquet) and **`charts`**
(simple charts). Every chart takes column names; pass `save=...` to save it or
`ax=...` to place it in a grid. Copy a chart from `charts.py` to make your own.

In [ ]:
from dreamerv4uwm.planning.experiments.study.analysis import dataset, charts, schema
%matplotlib inline
import pandas as pd; pd.set_option("display.width", 170)

# load a run's parquet (build it once from shards with dataset.build):
df = dataset.load("mcts_sweep/mcts_sweep_run1/analysis/trees_all.parquet")
# for a fresh run:  df = dataset.build("path/to/<run>/results", out="trees.parquet")
print(dataset.overview(df))
df.head(3)

## 1. Which knobs move the outcome?

`charts.response` plots mean +/- SEM of a column vs a knob. Wrap it in
`charts.ofat(df, knob)` for a clean one-factor sweep.

In [ ]:
charts.response(charts.ofat(df, "ctx_noise"), "ctx_noise", "g_random")

In [ ]:
# all the knobs at once
import matplotlib.pyplot as plt
knobs = ["horizon", "max_depth", "ctx_noise", "sim_horizon", "branching", "c_ucb"]
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, k in zip(axes.ravel(), knobs):
    charts.response(charts.ofat(df, k), k, "g_random", ax=ax)
fig.tight_layout()

## 2. What predicts success?

`charts.corr_bars` ranks the columns most correlated (Spearman) with the outcome.

In [ ]:
charts.corr_bars(df, "g_random")

In [ ]:
# the numbers behind it (a sorted Series):
charts.correlations(df, "g_random").tail(8)

## 3. The mechanism

Colour a scatter of two metrics by the outcome.

In [ ]:
charts.scatter(df, "edge_val_std", "val_std", color="g_random")

## 4. Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, col in zip(axes, ["g_random", "root_bci", "exploit_explore_ratio"]):
    charts.hist(df, col, ax=ax)
fig.tight_layout()

## 5. Make your own

A chart is a few lines of matplotlib on `df[col]`. Save with `save=...`.

In [ ]:
charts.scatter(df, "visit_entropy", "g_random", color="ctx_noise", save="my_chart.png")
# every column you can pass:
sorted(charts.metric_cols(df))

## Reference: what each column means

In [ ]:
with pd.option_context("display.max_rows", None, "display.max_colwidth", 90):
    display(schema.data_dictionary(df))

## Deck figures (optional)

The five polished figures the LaTeX deck embeds live in `plots.py`; `fig_*(df, None)` renders one inline.

In [ ]:
from dreamerv4uwm.planning.experiments.study.analysis import plots
plots.fig_ctxnoise(df, None)
# regenerate all deck PDFs:  plots.make_deck(df, "../presentation/figures")